# TIB FVOA Electrical Noise and Response

Standalone investigation of both attenuator drives, their amplifiers, and PD response. This notebook controls
PCB operating points through `hispec_fibpcb.py`, collects the TIB stream, and downloads the Siglent over the
network. It does not import or execute the commissioning notebook. Previous Copy1 scope analysis was untested;
its hypothetical response-time filters are not treated as measurements here.

Use the workspace `.venv` kernel and the installed `SCPI-Instrument-Control` package used by
[sigilent_demo.ipynb](sigilent_demo.ipynb). All new acquisition and analysis code is here. Run cells individually;
hardware actions start disabled. Set scale/timebase/probe attenuation on the front panel, then capture.

| Signal | Probe plane | Expected operating range |
|---|---|---|
| PD | ADC input, downstream of the PCB analog filter | 0–2 V |
| FVOA1, FVOA2 | Post-amplifier drive at each actuator | 0–5 V |
| DAC preamp | One selected amplifier input | 0–3.3 V |

Repeat with the preamp probe on the other DAC when useful. Use short, fast records to look for oscillation and
long records for low-frequency noise. Measure probe/instrument floors with the same bandwidth and vertical
settings. Record exact probe connections and ground arrangement; correlated pickup can mimic drive correlation.

In [1]:
from pathlib import Path
from dataclasses import asdict, is_dataclass, fields
from datetime import datetime, timezone
import math
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import display

TOOLS = next(p.resolve() for p in (Path.cwd(), Path.cwd() / 'tools', Path.cwd() / 'hispec-tib/tools')
             if (p / 'hispec_fibpcb.py').is_file())
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))
import hispec_fibpcb as hspcb

plt.rcParams.update({'axes.grid': True, 'grid.alpha': 0.18, 'figure.figsize': (11, 5)})

from importlib.metadata import version
from scpi_control import Oscilloscope

In [2]:
CHANNEL = 'yj'                         # Copy/change to 'hk' for the other channel.
LASERS = {'yj': ['1028y', '1270j', '1430yj'], 'hk': ['1430hk', '1510h', '2330k']}[CHANNEL]
BROKER, DEVICE = 'hispec.caltech.edu', 'hsfib-tib'
OUTPUT, FIBER = f'{CHANNEL}_ao', 'M'    # Match the physical patch before each experiment.
CONDITION = 'room'                     # e.g. 'room', 'chamber_-5C'; measured temperatures are also saved.
BUILD_LABEL = 'enter flashed build / autolevel policy'
STARTUP_HISTORY = 'enter cold bank / previously enabled / elapsed idle time'
PATCH_NOTES = 'enter actual fiber endpoints, jumpers, fixed attenuators, and meters'
FLASHED_ADC_SPS = 64                   # Owner-specified flashed setup, not queried by the current API.
DETECTOR_BANDWIDTH_HZ = {'yj': 20.0, 'hk': 500.0}[CHANNEL]
ADC_INPUT_CAPACITANCE_UF = 0.47
CONNECT = False
pcb = globals().get('pcb')

DATA_DIR = TOOLS / 'fvoa_scope_data'
CAPTURE_FILES = globals().get('CAPTURE_FILES', [])
SCOPE_HOST, SCOPE_PORT = '131.215.200.196', 5025
SIGNAL_CHANNELS = {'pd_mv':1, 'fvoa1_mv':2, 'fvoa2_mv':3, 'dac_pre_mv':4}
PREAMP_PHYSICAL = 'dac1'
PROBE_NOTES = 'enter probe type/attenuation, ground, bandwidth and floor-measurement details'
LASER = LASERS[0]

In [3]:
def parameter_table(values):
    """Flatten settings into a readable, typed table; no JSON or pickled objects."""
    rows = []
    def visit(key, value):
        if is_dataclass(value):
            value = asdict(value)
        if isinstance(value, dict):
            for name, item in value.items():
                visit(f'{key}.{name}' if key else str(name), item)
        elif isinstance(value, (tuple, list, np.ndarray)):
            for i, item in enumerate(value):
                visit(f'{key}.{i}', item)
        else:
            numeric = isinstance(value, (int, float, np.number)) and not isinstance(value, (bool, np.bool_))
            rows.append((key, 'number' if numeric else 'text', float(value) if numeric else np.nan,
                         '' if numeric or value is None else str(value)))
    visit('', values)
    return pd.DataFrame(rows, columns=['key', 'kind', 'number', 'text'])


def parameter(table, key, default=np.nan):
    """Read one saved scalar without reconstructing a driver object."""
    rows = table.loc[table.key.eq(key)]
    if rows.empty:
        return default
    row = rows.iloc[-1]
    return row.number if row.kind == 'number' else row.text


def table_records(frame):
    """Keep numeric dtypes; encode text and stream flag tuples as Unicode for allow_pickle=False."""
    if not len(frame.columns):
        return np.empty(len(frame), dtype=[])
    frame = frame.copy()
    string_types = {}
    for name in frame:
        if frame[name].dtype.kind == 'O' or isinstance(frame[name].dtype, pd.StringDtype):
            frame[name] = frame[name].map(lambda v: '|'.join(v) if isinstance(v, tuple) else '' if v is None else str(v))
            string_types[name] = f'U{max(1, frame[name].str.len().max() if len(frame) else 1)}'
    return frame.to_records(index=False, column_dtypes=string_types)


def save_tables(stem, **tables):
    """Archive raw acquisition before analysis. Exclusive creation never overwrites a capture."""
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    path = DATA_DIR / f'{stem}_{stamp}.npz'
    arrays = {name: table_records(value) if isinstance(value, pd.DataFrame) else np.asarray(value)
              for name, value in tables.items()}
    if any(a.dtype.hasobject for a in arrays.values()):
        raise TypeError('NPZ tables must have scalar numeric/Unicode columns, not Python objects.')
    with path.open('xb') as file:
        np.savez_compressed(file, **arrays)
    print(path)
    return path


def load_tables(path):
    """Offline read of this notebook's named arrays and structured tables."""
    with np.load(path, allow_pickle=False) as archive:
        return {key: pd.DataFrame.from_records(archive[key]) if archive[key].dtype.names is not None else archive[key].copy()
                for key in archive.files}


def snapshot(client, laser, *, output=None, fiber=None):
    """Query active settings and selected-laser telemetry; never request light or an all-bank PD read."""
    output, fiber = output or OUTPUT, fiber or FIBER
    source = f'{CHANNEL}_1430' if laser.startswith('1430') else f'{CHANNEL}_laser'
    values = dict(utc=datetime.now(timezone.utc).isoformat(), laser=laser, channel=CHANNEL,
        output=output, fiber=fiber, condition=CONDITION, build_label=BUILD_LABEL,
        startup_history=STARTUP_HISTORY, patch_notes=PATCH_NOTES, flashed_adc_sps=FLASHED_ADC_SPS,
        detector_bandwidth_hz=DETECTOR_BANDWIDTH_HZ, adc_input_capacitance_uf=ADC_INPUT_CAPACITANCE_UF,
        status=client.status(), clock=client.time(), heater=client.laser_bankheater(),
        bank=client.laser_bankpower(), laser_status=client.laser(laser), engineering=client.laser_status(laser),
        laser_settings=client.laser_settings(laser), coeff=client.atten_coeff(laser), atten=client.atten(laser),
        pd_settings=client.pd_settings(CHANNEL), dark=client.pd_dark(CHANNEL), switches=client.mems(),
        launch_loss=client.mems_route_loss(f'{source}_to_{output}'),
        return_loss=client.mems_route_loss(f'{CHANNEL}_{"mm" if fiber == "M" else "sm"}_to_{CHANNEL}_pd'))
    return parameter_table(values)

In [4]:
def open_stream(client, seconds, *, laser='none', fiber=None, output=None, autolevel=False, initial_level=None):
    """Stop an earlier owner BEFORE manual source setup; return a new finite-capacity collector.

    A former autolevel owner can shut down its source here. The returned manual/passive
    collector has no inherited shutdown obligation. Its stop leaves manually enabled lasers alone.
    """
    if not np.isfinite(seconds) or seconds <= 0:
        raise ValueError('Choose a positive capture duration.')
    client.stop_throughput(CHANNEL)
    return client.measure_throughput(laser, channel=CHANNEL, fiber=fiber or FIBER,
        output=(output or OUTPUT) if laser != 'none' else None, autolevel=autolevel,
        initial_level=initial_level, off_in_s=0, collect=True, format='binary',
        max_samples=max(1000, math.ceil(seconds / .05 * 1.5) + 1000))


def collect_trace(client, monitor, laser, seconds, label, *, context, extra=None, telemetry_s=2.0):
    """Block for an editable hold; save on completion or interrupt and detach the stream.

    Slow engineering queries give measured current/TEC context, not high-frequency current noise.
    Raw samples include the start transient; settling cuts belong in analysis. No automatic laser STOP.
    """
    started = time.monotonic()
    readings, error, completed = [], '', False
    try:
        next_read = started
        while time.monotonic() - started < seconds:
            if time.monotonic() >= next_read:
                before = time.time_ns() // 1_000_000
                status = client.laser_status(laser)
                ambient = client.status().amb_c
                heater = client.laser_bankheater()
                after = time.time_ns() // 1_000_000
                # PID configuration is already in the context table; each row keeps scalar telemetry.
                values = {k: v for k, v in asdict(status).items() if k != 'pid'}
                readings.append(dict(host_start_ms=before, host_end_ms=after, ambient_c=ambient,
                    heater_on=heater.heater_on, heater_auto_state=heater.auto_state, **values))
                next_read = time.monotonic() + telemetry_s
            time.sleep(min(.1, max(0, seconds - (time.monotonic() - started))))
        completed = True
    except BaseException as exc:
        error = f'{type(exc).__name__}: {exc}'
        raise
    finally:
        # Stop/detach first to include queued telemetry. Save even if the stop command fails.
        stop_error = ''
        try:
            monitor.stop()
        except Exception as exc:
            stop_error = f'{type(exc).__name__}: {exc}'
            raise
        finally:
            frame = monitor.to_dataframe()
            for name, value in dict(experiment=label, source_laser=laser, condition=CONDITION,
                                    build_label=BUILD_LABEL, **(extra or {})).items():
                frame[name] = value
            telemetry = pd.DataFrame(readings)
            outcome = parameter_table(dict(complete=completed, error=error, stop_error=stop_error, requested_s=seconds,
                elapsed_s=time.monotonic()-started, samples=len(frame), collector_capacity=monitor.max_samples,
                capacity_reached=len(frame) >= monitor.max_samples))
            path = save_tables(f'{label}_{laser}', samples=frame, telemetry=telemetry, context=context, outcome=outcome)
            CAPTURE_FILES.append(path)
    return path

In [5]:
def current_fraction(settings, requested_ma):
    """Translate one reachable Maiman setpoint through the existing fractional command API."""
    if requested_ma == 0:
        return 0.
    low = math.ceil((settings.threshold_current_ma + 1e-9)*10)/10
    high = math.floor((settings.nominal_current_ma + 1e-9)*10)/10
    if not low <= requested_ma <= high or not np.isclose(requested_ma*10, round(requested_ma*10), atol=1e-7):
        raise ValueError(f'{requested_ma} mA is not a reachable positive 0.1 mA setpoint in [{low}, {high}].')
    return (requested_ma-settings.threshold_current_ma)/(settings.nominal_current_ma-settings.threshold_current_ma)

In [6]:
def read_frozen_scope(scope):
    """Download synchronous WORD traces from one stopped acquisition; leave it stopped."""
    if scope.trigger.mode != 'STOP':
        raise RuntimeError('Stop acquisition before downloading the four channels.')
    channels = list(SIGNAL_CHANNELS.values())
    if sorted(channels) != [1,2,3,4]:
        raise ValueError('Assign each physical signal to a different scope channel 1–4.')
    settings = {name: scope.get_channel(channel).get_configuration() for name, channel in SIGNAL_CHANNELS.items()}
    if any(not cfg['enabled'] for cfg in settings.values()):
        raise RuntimeError('Enable all four selected channels and acquire a new record.')
    if any(cfg['coupling'] != 'DC' for cfg in settings.values()):
        raise ValueError('DC coupling is required for operating means and fractional optical response.')
    pending = {name: scope.waveform.acquire(channel, format='WORD', stride=1) for name, channel in SIGNAL_CHANNELS.items()}
    if scope.trigger.mode != 'STOP':
        raise RuntimeError('Scope restarted during transfer; the traces are not one synchronous record.')
    t = pending['pd_mv'].time
    if len(t)<3 or not np.all(np.isfinite(t)) or not np.all(np.diff(t)>0):
        raise ValueError('Invalid waveform time axis.')
    for name, wf in pending.items():
        if not np.array_equal(wf.time,t) or wf.voltage.shape != t.shape or not np.all(np.isfinite(wf.voltage)):
            raise ValueError(f'{name}: incomplete trace or a different acquisition/time axis.')
    traces = pd.DataFrame({'t_s': t, **{name: wf.voltage*1000 for name,wf in pending.items()}})
    metadata = parameter_table(dict(instrument=scope.device_info, settings=settings,
        trigger=scope.trigger.get_configuration(), timebase_s_per_div=scope.timebase,
        signal_channels=SIGNAL_CHANNELS, preamp_physical=PREAMP_PHYSICAL,
        package_version=version('SCPI-Instrument-Control'), transfer_format='WORD', stride=1,
        downloaded_utc=datetime.now(timezone.utc).isoformat(), probe_notes=PROBE_NOTES))
    return traces, metadata


def drive_slopes(context, mean_postamp_mv):
    """Captured-model local dA/dV, per post-amplifier mV, using the driver's model evaluator."""
    rows = []
    for physical, voltage in zip(('dac1','dac2'), mean_postamp_mv):
        coeff = tuple(parameter(context,f'coeff.{physical}.{key}') for key in
                      ('fvoa_50pct_mv','slope_inv_fvoa_mv','max_atten_db','gain'))
        corr = tuple(parameter(context,f'coeff.{physical}.correction_coeff.{i}') for i in range(6))
        limit = parameter(context,f'coeff.{physical}.max_calibrated_db')
        c = (*coeff,corr,limit)
        a = hspcb._atten_db_from_coeff(c,(voltage+np.array([-.1,0.,.1]))/coeff[3])
        slope = (a[2]-a[0])/.2
        rows.append(dict(physical=physical, postamp_mv=voltage, attenuation_db=a[1],
            calibrated=a[1]<=limit, slope_db_per_mv=slope, fractional_per_mv=-np.log(10)/10*slope))
    return pd.DataFrame(rows)

## Operating point

Query the installed coefficients and settings, then edit the current and DAC pair. No calibration is run here.
The setup cell quiesces any previous autolevel owner before manual source commands. It starts at the current
positive setpoint (otherwise minimum autolevel) and the presently applied FVOAs. Select a useful unsaturated
PD level before doing small-step or fractional-noise analysis. Full shutdown is a separate final cell.

The reported FVOA response range (5 ms minimum / 50 ms typical / 60 ms maximum) is not a measured one-pole
time constant. The detector bandwidths and ADC-input capacitor are recorded setup facts, not enough to infer
an end-to-end filter. The measured drive→ADC-input response below already includes the FVOA, detector and
PCB analog chain; do not apply those filters to the PD trace a second time.

In [10]:
if CONNECT:
    if pcb is None:
        pcb = hspcb.HispecFibPcb(BROKER,device=DEVICE,connect=True,auto_connect=False)
    elif not pcb.is_connected:
        pcb.connect()
    display(snapshot(pcb,LASER))

,key,kind,number,text
0,utc,text,NaN,2026-09-21T20:54:20.528847+00:00
1,laser,text,NaN,1028y
2,channel,text,NaN,yj
3,output,text,NaN,yj_ao
4,fiber,text,NaN,M
...,...,...,...,...
239,return_loss.lasers.4.name,text,NaN,1510h
240,return_loss.lasers.4.value,number,0.02,
241,return_loss.lasers.5.name,text,NaN,2330k
242,return_loss.lasers.5.value,number,0.02,


In [9]:
CONNECT=True

In [21]:
pcb.laser('1430yj', value=1)

CommandOk(status='ok')

In [ ]:
QUERY_OPERATING_POINT = False
if QUERY_OPERATING_POINT:
    settings,actual,drive = pcb.laser_settings(LASER),pcb.laser_status(LASER),pcb.atten(LASER)
    CURRENT_MA = actual.i_mA if actual.i_mA is not None and actual.i_mA>0 else settings.min_autolevel_current_ma
    DAC_MV = [drive.v1_mv,drive.v2_mv]
    display(pd.DataFrame([dict(laser=LASER,current_ma=CURRENT_MA,dac1_mv=DAC_MV[0],dac2_mv=DAC_MV[1],
                               tune_nm=settings.tune_nm,measured_current_ma=actual.curr_meas_ma)]))
# Edit CURRENT_MA and DAC_MV here after querying.

In [ ]:
APPLY_OPERATING_POINT = False
if APPLY_OPERATING_POINT:
    pcb.stop_throughput(CHANNEL)
    for name in LASERS:
        if name != LASER:
            pcb.laser(name,value=0,autooff_s=0)
    pcb.mems_route(f'{CHANNEL}_1430' if LASER.startswith('1430') else f'{CHANNEL}_laser',OUTPUT)
    pcb.mems_route(f'{CHANNEL}_{"mm" if FIBER=="M" else "sm"}',f'{CHANNEL}_pd')
    pcb.pd(CHANNEL)
    pcb.atten(LASER,value1_mv=DAC_MV[0],value2_mv=DAC_MV[1])
    pcb.laser(LASER,value=current_fraction(pcb.laser_settings(LASER),round(CURRENT_MA*10)/10),autooff_s=0)
    operating_file = save_tables(f'operating_point_{LASER}',context=snapshot(pcb,LASER))

## Stationary, oscillation, and instrument-floor records

For a new synchronized bench interval, select `NEW_RECORD=True`: the scope runs for the chosen duration and
then freezes. TIB streaming runs across that interval and download; the scope's relative timebase and board
clock are separate and are not phase-aligned by this procedure. For a manually frozen record, use
`NEW_RECORD=False`; its TIB samples are contextual, not claimed simultaneous.

Acquisition mode, bandwidth, timebase, enabled channels and probe factors must be appropriate on the front
panel. WORD transfer preserves full recorded resolution. It cannot restore analog bandwidth or uncropped
peaks. The downloaded trace duration is displayed; use it, rather than the wait duration, for analysis.
Set `CAPTURE_KIND='probe_floor'` for your manually arranged floor control and document what is terminated.

In [ ]:
RUN_SCOPE_CAPTURE = False
NEW_RECORD = True
RECORD_SECONDS = 20.0
CAPTURE_KIND = 'long_noise'             # Also 'fast_oscillation' or 'probe_floor'.
if RUN_SCOPE_CAPTURE:
    context = snapshot(pcb,LASER)
    monitor = open_stream(pcb,RECORD_SECONDS+60,laser=LASER)
    traces,scope_settings = pd.DataFrame(),parameter_table({})
    error = ''
    acquisition_start_ms = time.time_ns()//1_000_000
    try:
        with Oscilloscope(SCOPE_HOST,port=SCOPE_PORT,timeout=30.) as osc:
            if NEW_RECORD:
                osc.run()
                time.sleep(RECORD_SECONDS)
                osc.stop()
            traces,scope_settings = read_frozen_scope(osc)
    except BaseException as exc:
        error = f'{type(exc).__name__}: {exc}'
        raise
    finally:
        try:
            monitor.stop()
        finally:
            path = save_tables(f'{CAPTURE_KIND}_{LASER}',waveforms=traces,scope=scope_settings,
                samples=monitor.to_dataframe(),context=context,
                acquisition=parameter_table(dict(kind=CAPTURE_KIND,new_record=NEW_RECORD,error=error,
                    host_start_ms=acquisition_start_ms,host_end_ms=time.time_ns()//1_000_000)))
            CAPTURE_FILES.append(path)
    display(pd.DataFrame([dict(samples=len(traces),span_s=traces.t_s.iloc[-1]-traces.t_s.iloc[0],
                               sample_hz=1/np.median(np.diff(traces.t_s)))]))

In [ ]:
# Works for dark/probe-floor records as well as illuminated data; no fractional normalization is needed.
SCOPE_INSPECT_FILE = None
DISPLAY_WINDOW_S = None                 # e.g. (0., .001) to resolve a fast oscillation.
RAW_PSD_SEGMENT_S = .1                  # Increase for long low-frequency records.
if SCOPE_INSPECT_FILE is not None:
    inspection = load_tables(SCOPE_INSPECT_FILE)
    raw = inspection['waveforms']
    display(inspection['scope'])
    if not raw.empty:
        display(raw.drop(columns='t_s').agg(['mean','std','min','max']).T)
        fs = 1/np.median(np.diff(raw.t_s))
        view = raw if DISPLAY_WINDOW_S is None else raw.loc[raw.t_s.between(*DISPLAY_WINDOW_S)]
        stride = max(1,len(view)//20000)
        fig,axes = plt.subplots(4,2,figsize=(13,10),layout='constrained')
        for (name,channel),(trace_ax,psd_ax) in zip(SIGNAL_CHANNELS.items(),axes):
            trace_ax.plot(view.t_s.iloc[::stride],view[name].iloc[::stride],lw=.7)
            trace_ax.set(xlabel='Scope time (s)',ylabel=f'{name} (mV)')
            f,power = signal.welch(raw[name].to_numpy(),fs=fs,nperseg=min(len(raw),max(8,int(RAW_PSD_SEGMENT_S*fs))))
            psd_ax.loglog(f,np.sqrt(power));psd_ax.set(xlabel='Hz',ylabel='mV/√Hz')
        fig.suptitle(f'{Path(SCOPE_INSPECT_FILE).name} · waveform display stride {stride}; PSD uses full record')
        # A thinned full-record waveform can alias; reduce DISPLAY_WINDOW_S to inspect a fast cycle.

## Small repeated FVOA steps

Set an edge-triggerable timebase with a pretrigger baseline and enough posttrigger time for the PD plateau
(e.g. 0.5 s total to start; lengthen if needed). This cell changes trigger source, slope and level, arms single
acquisition, and changes **one DAC at a time**. Both positive and negative steps return to the same operating
point between captures. Other FVOA/laser settings remain fixed. Choose step sizes clearly above probe noise
but small enough for local response; compare two sizes if linearity is uncertain.

The level initially uses the installed amplifier gain; edit `TRIGGER_LEVEL_V` for a measured offset. Acquisition
must trigger and stop; a timeout archives available TIB data and the error. It does not force a trigger and
pretend a response was measured. Each completed four-channel record is archived before the next step.

In [ ]:
RUN_DRIVE_STEPS = False
STEP_DAC_MV = [2.]                     # Add a second small amplitude to test local linearity.
STEP_REPEATS = 3
STEP_BASELINE_SECONDS = 1.0
TRIGGER_WAIT_SECONDS = 10.0
TRIGGER_ARM_SECONDS = .3               # Must fill the selected pretrigger memory.
TRIGGER_LEVEL_V = None                 # None: midpoint from commanded DAC and installed gain.
STEP_FILES = globals().get('STEP_FILES', [])
if RUN_DRIVE_STEPS:
    for physical in ('dac1','dac2'):
        index = 0 if physical=='dac1' else 1
        signal_name = 'fvoa1_mv' if index==0 else 'fvoa2_mv'
        for amplitude in STEP_DAC_MV:
            for repeat in range(STEP_REPEATS):
                for sign in (1,-1):
                    target = list(DAC_MV)
                    target[index] += sign*amplitude
                    if not 0 <= target[index] <= 3300:
                        raise ValueError('Step exceeds DAC range.')
                    monitor = open_stream(pcb,TRIGGER_WAIT_SECONDS+60,laser=LASER)
                    traces,scope_settings = pd.DataFrame(),parameter_table({})
                    context,error = snapshot(pcb,LASER),''
                    command_times = {}
                    try:
                        pcb.atten(LASER,value1_mv=DAC_MV[0],value2_mv=DAC_MV[1])
                        time.sleep(STEP_BASELINE_SECONDS)
                        with Oscilloscope(SCOPE_HOST,port=SCOPE_PORT,timeout=30.) as osc:
                            osc.stop()
                            osc.trigger.set_edge_trigger(source=f'C{SIGNAL_CHANNELS[signal_name]}',slope='POS' if sign>0 else 'NEG')
                            gain = parameter(context,f'coeff.{physical}.gain')
                            osc.trigger.level = TRIGGER_LEVEL_V if TRIGGER_LEVEL_V is not None else (DAC_MV[index]+sign*amplitude/2)*gain/1000
                            osc.trigger.single()
                            time.sleep(TRIGGER_ARM_SECONDS)
                            command_times['host_start_ms'] = time.time_ns()//1_000_000
                            applied = pcb.atten(LASER,value1_mv=target[0],value2_mv=target[1])
                            command_times['host_end_ms'] = time.time_ns()//1_000_000
                            deadline = time.monotonic()+TRIGGER_WAIT_SECONDS
                            while osc.trigger.mode != 'STOP':
                                if time.monotonic()>=deadline:
                                    osc.stop()
                                    raise TimeoutError('No completed step trigger; review source, level and record length.')
                                time.sleep(.05)
                            traces,scope_settings = read_frozen_scope(osc)
                    except BaseException as exc:
                        error = f'{type(exc).__name__}: {exc}'
                        raise
                    finally:
                        try:
                            monitor.stop()
                        finally:
                            path = save_tables(f'step_{LASER}_{physical}',waveforms=traces,scope=scope_settings,
                                samples=monitor.to_dataframe(),context=context,
                                step=parameter_table(dict(physical=physical,sign=sign,amplitude_dac_mv=amplitude,
                                    repeat=repeat,error=error,**command_times)))
                            STEP_FILES.append(path)
                            pcb.atten(LASER,value1_mv=DAC_MV[0],value2_mv=DAC_MV[1])

In [ ]:
def scope_spectra(frame, context, nperseg=16384):
    """Synchronous drive/PD spectra retaining complex cross terms; no optical or ADC refiltering."""
    t = frame.t_s.to_numpy(float)
    dt = np.diff(t)
    if not np.allclose(dt, np.median(dt), rtol=1e-4, atol=1e-12):
        raise ValueError('Welch analysis requires the uniform scope timebase.')
    dark = parameter(context,'dark.dark.mean_mv',0.)
    pd_net = frame.pd_mv.to_numpy(float)-dark
    if np.mean(pd_net)<=0:
        raise ValueError('Use positive illuminated PD net signal for fractional spectra.')
    y = pd_net/np.mean(pd_net)-1
    x1,x2 = frame.fvoa1_mv.to_numpy(float),frame.fvoa2_mv.to_numpy(float)
    n = min(nperseg,len(t))
    kw = dict(fs=1/np.median(dt),nperseg=n,detrend='constant')
    f,p1 = signal.welch(x1,**kw)
    _,p2 = signal.welch(x2,**kw)
    _,py = signal.welch(y,**kw)
    _,pre = signal.welch(frame.dac_pre_mv.to_numpy(float),**kw)
    _,cross = signal.csd(x1,x2,**kw)       # scipy uses conjugate(X1) * X2.
    _,c1y = signal.coherence(x1,y,**kw)
    _,c2y = signal.coherence(x2,y,**kw)
    averages = 1+(len(t)-n)//(n-n//2)
    if averages < 2:
        c1y[:],c2y[:] = np.nan,np.nan   # Single-periodogram coherence is identically one, not evidence.
    return pd.DataFrame(dict(f_hz=f,drive1_mv2_per_hz=p1,drive2_mv2_per_hz=p2,
        preamp_mv2_per_hz=pre,pd_fraction2_per_hz=py,cross_real=cross.real,cross_imag=cross.imag,
        coherence_1_pd=c1y,coherence_2_pd=c2y,welch_averages=averages))


def propagated_drive_psd(p1,p2,cross,h1,h2):
    """Two measured complex responses with correlated drive noise; preserves cancellation as well as addition."""
    return np.abs(h1)**2*p1 + np.abs(h2)**2*p2 + 2*np.real(np.conj(h1)*h2*cross)


def supported_integral(f, power, usable):
    """Integrate only adjacent valid bins, never draw across unsupported frequency gaps."""
    adjacent = usable[:-1] & usable[1:]
    return np.sum((power[:-1]+power[1:])[adjacent]*np.diff(f)[adjacent]/2)

In [ ]:
def measured_step_fft(frame, physical, *, pre=(-.08,-.01), post=(.15,.25), analysis_hz=2000.):
    """Estimate drive→fractional-PD response from an observed small step, without a time-constant model.

    The same differentiation/window is applied to drive and PD. End windows must bracket settled plateaus.
    Anti-aliased decimation makes short optical response analysis independent of a MHz acquisition rate.
    Repeat both step signs and inspect the waveforms; this estimator cannot make a non-small step linear.
    """
    column = 'fvoa1_mv' if physical=='dac1' else 'fvoa2_mv'
    t = frame.t_s.to_numpy(float)
    dt = np.median(np.diff(t))
    if not np.allclose(np.diff(t),dt,rtol=1e-4,atol=1e-12):
        raise ValueError('Response timebase must be uniform.')
    use = (t>=pre[0]) & (t<=post[1])
    t = t[use]
    signals = frame.loc[use,[column,'pd_mv']].to_numpy(float)
    down = max(1,int(round(1/(dt*analysis_hz))))
    if down>1:
        signals = signal.resample_poly(signals,1,down,axis=0,padtype='line')
        t = t[0]+np.arange(len(signals))*dt*down
    before,after = (t>=pre[0]) & (t<=pre[1]), (t>=post[0]) & (t<=post[1])
    if before.sum()<4 or after.sum()<4:
        raise ValueError('The saved record does not cover the chosen baseline and final windows.')
    v0,p0 = signals[before].mean(axis=0)
    v1,p1 = signals[after].mean(axis=0)
    dv = v1-v0
    if dv==0 or p0<=0:
        raise ValueError('Need a resolved drive step and positive dark-subtracted PD level.')
    # Input to this function has already had the saved dark subtracted from pd_mv.
    ddrive = np.diff(signals[:,0])
    dfrac = np.diff(signals[:,1]/p0)
    frequency = np.fft.rfftfreq(len(ddrive),dt*down)
    x,y = np.fft.rfft(ddrive),np.fft.rfft(dfrac)
    trace = pd.DataFrame(dict(t_s=t,drive_step=(signals[:,0]-v0)/dv,
                              pd_step=(signals[:,1]-p0)/(p1-p0) if p1!=p0 else np.nan))
    info = dict(drive_step_mv=dv,pd_step_mv=p1-p0,dc_fractional_per_mv=(p1-p0)/p0/dv,
                baseline_pd_mv=p0,final_pd_mv=p1,sample_hz=1/(dt*down),duration_s=t[-1]-t[0])
    return frequency,x,y,trace,info

## Offline response estimation

Set baseline/final windows after inspecting the repeated drive and PD traces. The Fourier ratio of differentiated
steps estimates the complex **post-amplifier drive → fractional PD** response; no exponential is imposed.
Keep step signs and amplitudes visible. Repeated-step coherence is a repeatability diagnostic here, not proof
of causation in later stationary noise. Frequency support requires adequate input excitation and repeatability;
frequencies above the editable limit are not extrapolated. The near-DC step sensitivity is compared with the
calibrated local slope. Disagreement can expose nonlinearity, drift, electrical scale errors or calibration error.

In [ ]:
RESPONSE_FILES = list(STEP_FILES) if 'STEP_FILES' in globals() else []
PRE_WINDOW_S, POST_WINDOW_S = (-.08,-.01),(.15,.25)
RESPONSE_MAX_HZ = 10.
RESPONSE_COHERENCE_MIN = .8
response_parts,step_rows,response_context_rows = [],[],[]
response_grid = None
if RESPONSE_FILES:
    response_figure, response_axis = plt.subplots(figsize=(10,4))
for path in RESPONSE_FILES:
    saved = load_tables(path)
    if saved['waveforms'].empty or parameter(saved['step'],'error',''):
        continue
    frame,context = saved['waveforms'].copy(),saved['context']
    response_context_rows.append(context.assign(file=str(path)))
    physical = parameter(saved['step'],'physical')
    frame['pd_mv'] -= parameter(context,'dark.dark.mean_mv',0.)
    f,x,y,trace,info = measured_step_fft(frame,physical,pre=PRE_WINDOW_S,post=POST_WINDOW_S)
    if response_grid is None:
        response_grid = f[f<=RESPONSE_MAX_HZ]
    # Interpolation never expands a capture's frequency support. Each point still needs repeated evidence.
    valid = (response_grid>=f.min()) & (response_grid<=min(f.max(),RESPONSE_MAX_HZ))
    q = response_grid[valid]
    xi = np.interp(q,f,x.real)+1j*np.interp(q,f,x.imag)
    yi = np.interp(q,f,y.real)+1j*np.interp(q,f,y.imag)
    response_parts.append(pd.DataFrame(dict(file=str(path),physical=physical,f_hz=q,
        xx=np.abs(xi)**2,yy=np.abs(yi)**2,xy_real=(np.conj(xi)*yi).real,xy_imag=(np.conj(xi)*yi).imag)))
    slopes = drive_slopes(context,[frame.fvoa1_mv.mean(),frame.fvoa2_mv.mean()]).set_index('physical')
    step_rows.append(dict(file=str(path),physical=physical,sign=parameter(saved['step'],'sign'),
        amplitude_dac_mv=parameter(saved['step'],'amplitude_dac_mv'),expected_fractional_per_mv=slopes.loc[physical,'fractional_per_mv'],**info))
    response_axis.plot(trace.t_s,trace.pd_step,lw=.8,label=f'{physical}, {info["drive_step_mv"]:+.2f} mV')
if step_rows:
    response_axis.set(xlabel='Scope time (s)',ylabel='Normalized PD step'); response_axis.legend(fontsize=7)
    display(pd.DataFrame(step_rows))
response_model = pd.DataFrame(columns=['physical','f_hz','h_real','h_imag','coherence','usable','repeats'])
if response_parts:
    ensemble = pd.concat(response_parts,ignore_index=True)
    response_rows = []
    for (physical,frequency),group in ensemble.groupby(['physical','f_hz']):
        xx,yy = group.xx.mean(),group.yy.mean()
        xy = group.xy_real.mean()+1j*group.xy_imag.mean()
        h = xy/xx if xx>0 else complex(np.nan,np.nan)
        coherence = abs(xy)**2/(xx*yy) if xx*yy>0 else np.nan
        floor = ensemble.loc[ensemble.physical.eq(physical)].groupby('f_hz').xx.mean().max()*.04
        response_rows.append(dict(physical=physical,f_hz=frequency,h_real=h.real,h_imag=h.imag,
            coherence=coherence,usable=len(group)>=3 and coherence>=RESPONSE_COHERENCE_MIN and xx>=floor,repeats=len(group)))
    response_model = pd.DataFrame(response_rows)
    display(response_model)
    for physical,g in response_model.groupby('physical'):
        plt.figure(figsize=(9,3)); plt.plot(g.f_hz,np.hypot(g.h_real,g.h_imag),'o-',label=physical)
        plt.scatter(g.loc[~g.usable,'f_hz'],np.hypot(g.loc[~g.usable,'h_real'],g.loc[~g.usable,'h_imag']),marker='x',color='red',label='unsupported')
        plt.xlabel('Frequency (Hz)'); plt.ylabel('|fractional PD / drive mV|'); plt.legend()
response_contexts = pd.concat(response_context_rows,ignore_index=True) if response_context_rows else pd.DataFrame()
SAVE_RESPONSE = False
if SAVE_RESPONSE:
    response_file = save_tables('measured_response',response=response_model,steps=pd.DataFrame(step_rows),
        contexts=response_contexts,analysis_settings=parameter_table(dict(pre_window_s=PRE_WINDOW_S,post_window_s=POST_WINDOW_S,
            max_hz=RESPONSE_MAX_HZ,coherence_min=RESPONSE_COHERENCE_MIN)))

## Spectra, correlation and expected throughput-sample variation

The two-drive prediction retains the complex cross-spectrum:
`S_pred = |H1|² S11 + |H2|² S22 + 2 Re(conj(H1) H2 S12)` with SciPy's `S12 = conj(X1) X2` convention.
Correlated voltages can reinforce or cancel optical variation. The response must be measured at the same
current, attenuation, optical path and environmental state as the noise record; use the saved context to check.
Large deviations from a local linear response, probe floors, or weak excitation limit this estimate.

Compare measured PD spectra against this contribution, retaining other laser/detector/electrical noise.
A static calibrated-slope calculation is shown only as a low-frequency sensitivity comparison. Do not turn
full-band drive RMS into optical RMS using a static slope. The variance table flags ADC-input overrange; the linear conversion prediction is invalid when clipping occurs.
Coherence is left undefined when a record supplies only one Welch segment.
Scope/TIB records here lack a common trigger/clock,
so no cross-instrument coherence is claimed.

For an individual throughput sample, the ADC's conversion transfer matters; the 50 ms publication period is
not an integration window. `ADC_RESPONSE_FILE` can supply measured/verified frequency response at the saved
ADC rate (table `adc_response`: `f_hz`, `magnitude`; table `context`: `adc_sps`, `provenance`). Without it, report
ADC-input variance and a clearly labeled **1/64 s boxcar scenario**, not a verified ADS1115 filter. Integration
includes only supported frequencies and does not silently treat unmeasured frequencies as zero. Aliasing moves
power into the telemetry band; do not discard all physical frequencies above 10 Hz before computing a
single-sample variance. This first study does not predict configurable-window/autocalibration averages.

In [ ]:
NOISE_FILE = None                       # New scope capture NPZ, no hardware needed.
RESPONSE_FILE = None                    # Optional saved measured_response NPZ; otherwise use table above.
FLOOR_FILE = None                       # Same-probe/scale/bandwidth instrument-floor record.
ADC_RESPONSE_FILE = None
WELCH_SEGMENT_SECONDS = 2.0             # Increase on long records to resolve slower noise.
if RESPONSE_FILE is not None:
    response_archive = load_tables(RESPONSE_FILE)
    response_model, response_contexts = response_archive['response'], response_archive['contexts']
if NOISE_FILE is not None:
    saved = load_tables(NOISE_FILE)
    frame,context = saved['waveforms'],saved['context']
    # Display settings alongside the response provenance before interpreting the propagated spectrum.
    if not response_contexts.empty:
        compare_keys = ['laser','condition','output','fiber','patch_notes','laser_status.i_mA','atten.v1_mv','atten.v2_mv','flashed_adc_sps']
        comparison = pd.concat([response_contexts,context.assign(file='NOISE CAPTURE')],ignore_index=True)
        display(comparison.loc[comparison.key.isin(compare_keys)].pivot_table(index='key',columns='file',values=['number','text'],aggfunc='first'))
        for key in compare_keys:
            for file,old_context in response_contexts.groupby('file',sort=False):
                old,new = parameter(old_context,key),parameter(context,key)
                equal = np.isclose(old,new,rtol=1e-6,atol=.01,equal_nan=True) if isinstance(old,(int,float,np.number)) and isinstance(new,(int,float,np.number)) else old==new
                if not equal:
                    raise ValueError(f'Response operating point differs at {key}: {file}. Select matching response/noise captures.')
    fs = 1/np.median(np.diff(frame.t_s))
    psd = scope_spectra(frame,context,nperseg=max(8,int(WELCH_SEGMENT_SECONDS*fs)))
    f = psd.f_hz.to_numpy(float)
    display(drive_slopes(context,[frame.fvoa1_mv.mean(),frame.fvoa2_mv.mean()]))
    display(frame.drop(columns='t_s').agg(['mean','std','min','max']).T)
    display(frame.drop(columns='t_s').corr())
    fig,axes = plt.subplots(3,1,figsize=(11,9),layout='constrained')
    stride = max(1,len(frame)//20000)     # Display decimation only; analysis/archive use all samples.
    axes[0].plot(frame.t_s.iloc[::stride],frame.pd_mv.iloc[::stride],lw=.7)
    axes[0].set(xlabel='Scope time (s)',ylabel='ADC-input PD (mV)')
    for label,col in [('FVOA1','drive1_mv2_per_hz'),('FVOA2','drive2_mv2_per_hz'),('DAC preamp','preamp_mv2_per_hz')]:
        axes[1].loglog(f,np.sqrt(psd[col]),label=label)
    axes[1].set(xlabel='Hz',ylabel='Drive density (mV/√Hz)'); axes[1].legend()
    axes[2].loglog(f,np.sqrt(psd.pd_fraction2_per_hz),label='Measured fractional PD')
    response_h,usable = {},np.ones(len(f),dtype=bool)
    for physical in ('dac1','dac2'):
        g = response_model.loc[response_model.physical.eq(physical)].sort_values('f_hz')
        h = np.full(len(f),complex(np.nan,np.nan))
        supported = np.zeros(len(f),dtype=bool)
        # Interpolate only between adjacent supported measured-response bins.
        for i in range(len(g)-1):
            a,b = g.iloc[i],g.iloc[i+1]
            if not (a.usable and b.usable):
                continue
            mask = (f>=a.f_hz)&(f<=b.f_hz)
            h[mask] = np.interp(f[mask],[a.f_hz,b.f_hz],[a.h_real,b.h_real])+1j*np.interp(f[mask],[a.f_hz,b.f_hz],[a.h_imag,b.h_imag])
            supported |= mask
        response_h[physical]=h
        usable &= supported
    cross = psd.cross_real.to_numpy()+1j*psd.cross_imag.to_numpy()
    predicted = propagated_drive_psd(psd.drive1_mv2_per_hz.to_numpy(),psd.drive2_mv2_per_hz.to_numpy(),
                                     cross,response_h['dac1'],response_h['dac2'])
    if np.any(predicted[usable]<-1e-12):
        raise ValueError('Negative predicted PSD: inspect cross-spectrum/response consistency.')
    predicted=np.maximum(predicted,0)
    psd['predicted_fraction2_per_hz'],psd['response_supported']=predicted,usable
    axes[2].loglog(f[usable],np.sqrt(predicted[usable]),label='Both measured drive responses + correlation')
    axes[2].set(xlabel='Hz',ylabel='Fractional PD / √Hz'); axes[2].legend()
    if FLOOR_FILE is not None:
        floor = load_tables(FLOOR_FILE)
        floor_frame = floor['waveforms']
        floor_fs = 1/np.median(np.diff(floor_frame.t_s))
        ff,fp = signal.welch(floor_frame.pd_mv.to_numpy(),fs=floor_fs,
                            nperseg=min(len(floor_frame),max(8,int(WELCH_SEGMENT_SECONDS*floor_fs))))
        mean_net = frame.pd_mv.mean()-parameter(context,'dark.dark.mean_mv',0.)
        axes[2].loglog(ff,np.sqrt(fp)/mean_net,':',label='PD probe floor / illuminated net mean'); axes[2].legend()
    adc_sps = parameter(context,'flashed_adc_sps')
    scenarios = {'ADC input; before conversion': np.ones(len(f)),
                 f'{adc_sps:g} SPS boxcar scenario (unverified ADC model)': np.sinc(f/adc_sps)**2}
    if ADC_RESPONSE_FILE is not None:
        adc=load_tables(ADC_RESPONSE_FILE)
        if not np.isclose(parameter(adc['context'],'adc_sps'),adc_sps):
            raise ValueError('ADC response rate does not match capture.')
        a=adc['adc_response'].sort_values('f_hz')
        scenarios['Verified ADC response: '+str(parameter(adc['context'],'provenance'))] = np.interp(f,a.f_hz,a.magnitude,left=np.nan,right=np.nan)**2
    variance_rows=[]
    for label,weight in scenarios.items():
        support=usable & np.isfinite(weight)
        variance_rows.append(dict(model=label,adc_input_overrange_fraction=float((frame.pd_mv>=hspcb.PD_ADC_USABLE_MV).mean()),first_supported_hz=f[support].min() if support.any() else np.nan,
            last_supported_hz=f[support].max() if support.any() else np.nan,
            supported_bins=int(support.sum()),
            predicted_band_rms=np.sqrt(supported_integral(f,predicted*weight,support)) if support.sum()>1 else np.nan,
            measured_same_band_rms=np.sqrt(supported_integral(f,psd.pd_fraction2_per_hz.to_numpy()*weight,support)) if support.sum()>1 else np.nan))
    scope_variance=pd.DataFrame(variance_rows)
    scope_variance['unexplained_band_variance'] = scope_variance.measured_same_band_rms**2-scope_variance.predicted_band_rms**2
    display(scope_variance)
    tib=saved['samples']
    display(pd.DataFrame([dict(tib_samples=len(tib),mean_mv=tib.pd_net_mv.mean() if len(tib) else np.nan,
        rms_mv=tib.pd_net_mv.std() if len(tib) else np.nan,
        comment='Whole-record scatter includes drift/transients; compare stationary subsets below.')]))

In [ ]:
def stationary_segments(frame, settle_s=1.0, minimum_s=2.0):
    """Keep separate contiguous fixed-setting intervals; do not close gaps by dropping bad rows."""
    if len(frame) < 3:
        return []
    frame = frame.reset_index(drop=True)
    dt = frame.t_ms.diff()/1000
    nominal = dt[dt > 0].median()
    valid = np.isfinite(frame.pd_net_mv) & np.isfinite(frame.pd_mv) & frame.pd_mv.lt(hspcb.PD_ADC_USABLE_MV)
    valid &= ~frame.autolevel.astype(bool)
    boundary = (dt <= 0) | (dt > 1.5*nominal) | ~valid | ~valid.shift(fill_value=False)
    for column in ('channel', 'laser', 'experiment', 'point', 'step', 'step_index', 'selection'):
        if column in frame:
            labels = frame[column].astype('string').fillna('')
            boundary |= labels.ne(labels.shift(fill_value=''))
    # 0.1 mA steps are the experiment: never merge them using the old 0.2 mA tolerance.
    for column, tolerance in (('laser_current_ma', .049), ('atten_db', .005)):
        boundary |= frame[column].diff().abs().gt(tolerance)
    parts = []
    for _, group in frame.groupby(boundary.cumsum(), sort=False):
        if not valid.loc[group.index].all():
            continue
        group = group.loc[(group.t_ms-group.t_ms.iloc[0])/1000 >= settle_s]
        if len(group) >= 3 and (group.t_ms.iloc[-1]-group.t_ms.iloc[0])/1000 >= minimum_s:
            parts.append(group.reset_index(drop=True))
    return parts


def noise_statistics(part):
    """Empirical scatter, drift, spectrum, ACF and averaging; shared calibration errors are separate."""
    y = part.pd_net_mv.to_numpy(float)
    t = (part.t_ms.to_numpy(float)-float(part.t_ms.iloc[0]))/1000
    dt, rms = np.diff(t), np.std(y, ddof=1)
    fs = 1/np.median(dt)
    centered = y-y.mean()
    acf = signal.correlate(centered, centered, mode='full', method='fft')[len(y)-1:]/np.arange(len(y), 0, -1)
    acf = acf/acf[0] if acf[0] > 0 else np.full_like(acf, np.nan)
    f, psd = signal.welch(y, fs=fs, nperseg=min(1024, max(3, len(y)//4)), detrend='constant')
    blocks = []
    for count in np.unique(np.maximum(1, np.round(np.array([.05,.1,.25,.5,1,2,5,10,20,30,60])*fs).astype(int))):
        n = len(y)//count
        if n < 4:
            continue
        means = y[:n*count].reshape(n, count).mean(axis=1)
        blocks.append(dict(tau_s=count/fs, blocks=n, rms_mean_mv=np.std(means, ddof=1),
            iid_rms_mean_mv=rms/np.sqrt(count), allan_mv=np.sqrt(.5*np.mean(np.diff(means)**2)),
            adjacent_block_corr=np.corrcoef(means[:-1], means[1:])[0,1] if np.std(means)>0 else np.nan))
    metrics = dict(samples=len(y), duration_s=t[-1], mean_mv=y.mean(), rms_mv=rms,
        detrended_rms_mv=np.std(signal.detrend(y), ddof=1), drift_mv_per_s=np.polyfit(t, y, 1)[0],
        relative_rms=rms/abs(y.mean()) if abs(y.mean()) > 3*rms else np.nan,
        adjacent_corr=acf[1], sample_hz=fs, max_gap_ms=1000*max(dt), timing_jitter_ms=1000*np.std(dt))
    return metrics, pd.DataFrame(dict(f_hz=f, psd_mv2_per_hz=psd)), pd.DataFrame(blocks), acf

In [ ]:
if NOISE_FILE is not None:
    tib_parts=stationary_segments(saved['samples'],settle_s=1.,minimum_s=2.)
    tib_rows=[]
    for i,part in enumerate(tib_parts):
        metrics,tib_psd,blocks,acf=noise_statistics(part)
        tib_rows.append(dict(interval=i,**metrics))
    display(pd.DataFrame(tib_rows))
    plt.figure(figsize=(10,3))
    plt.plot(psd.f_hz,psd.coherence_1_pd,label='FVOA1–PD')
    plt.plot(psd.f_hz,psd.coherence_2_pd,label='FVOA2–PD')
    plt.xlim(0,min(100,psd.f_hz.max()));plt.ylim(0,1)
    plt.xlabel('Hz');plt.ylabel('Magnitude-squared coherence');plt.legend()
SAVE_NOISE_ANALYSIS=False
if SAVE_NOISE_ANALYSIS and NOISE_FILE is not None:
    save_tables('scope_analysis',spectra=psd,response=response_model,variance=scope_variance,
        tib_summary=pd.DataFrame(tib_rows),context=context,source=np.array(str(NOISE_FILE)))

## Interpretation and further measurements

Compare measured drive noise, amplifier input/output correlation, probe floors, repeated response signs/sizes,
and PD noise at the same operating point. A residual unexplained spectrum can contain laser noise, detector
noise, pickup, drift, or a wrong response estimate. Prior quiet FVOA-bypass laser measurements are useful
context; they do not justify assigning every fluctuation to the drives.

Record response/noise provenance before reusing an estimate at another current, attenuation, temperature,
probe configuration, or firmware ADC setting. Predictions above are band-limited contributions, not complete
noise budgets. The firmware's independent 10 mV-drive assumption and static calibration residual uncertainty
can then be evaluated against the data. Rehome a calculation into HISPEC or a future wrapper only after its
measurement assumptions have been verified.

In [ ]:
ZERO_AND_REMAIN_ENABLED=False
FULL_SHUTDOWN=False
if ZERO_AND_REMAIN_ENABLED or FULL_SHUTDOWN:
    try:
        pcb.stop_throughput(CHANNEL)
    finally:
        if FULL_SHUTDOWN:
            pcb.laser(LASER,stop=True)
        else:
            pcb.laser(LASER,value=0,autooff_s=0)
    save_tables(f'final_state_{LASER}',context=snapshot(pcb,LASER))

## Offline verification of this notebook

Notebook syntax/schema and offline execution were checked independently. Synthetic signed steps with a known
response checked the numerical response estimator; correlated drives checked reinforcement and phase-sensitive
cancellation. The complete response/noise/ADC-scenario analysis and NPZ provenance round trip were exercised.
A scope double checked synchronous WORD-record assembly and rejection of mismatched time axes. No live scope
or PCB acquisition was performed. Triggering, probe floors, linearity, bandwidth and ADC-response provenance
need bench verification; the illustrative boxcar is not a validated ADS1115 filter.